# Multilingual Functional Evaluation of the Solace Pipeline

## 1 Overview

In the four languages that Solace supports English, Malay, Chinese, and Tamil this notebook offers a functional assessment of the multimodal pipeline. In contrast to the Copenhagen Burnout Inventory validation, which asks whether the wellbeing score is numerically accurate when compared to an external instrument, this notebook poses a different but no less important question: does the pipeline function end-to-end and in every language on an actual recording? Solace's multilingual functioning is an explicit feature, and each feature mentioned in the report needs to be backed up by proof that it functions this notebook serves as that proof. A transcript generated using Whisper [1], a text-emotion score, an audio-emotion score, a facial-emotion score for video, a fused wellbeing score, and a recommendation translated into the chosen non-English language using NLLB [2] are all produced by running an actual recording through the application's production pipeline.

Throughout the project, there is a purposeful separation between accuracy validation and functional evaluation. The functional evaluation described here can span all four supported languages and show that the localisation path is exercised rather than just included in the code because it only needs a real recording in each language. Before making any conclusions, the evaluation reports its own coverage and is only based on the recordings that are stored on disk and arranged into a single folder for each language. This is significant since there is an inconsistent amount of material available actual user recordings are available for Tamil and English, only one recording is available for Chinese, and coverage in Malay may still be lacking at the time of a particular run. A language without a recording is presented as having no sample available rather than vanishing from the results since the notebook finds any recordings that are there and clearly specifies the coverage, as opposed to subtly excluding the thinner languages. As more recordings are recorded and the notebook is rerun, the result is a true but suitably scoped description of functional multilingual functioning, whose strength naturally increases.

## 2 Importing Libraries

The modest toolchain that is necessary for the functional evaluation is imported in this part. While sys is imported so that the project root can be placed on the import path in the following section and the Solace pipeline imported like any installed package, the standard-library modules offer filesystem traversal through pathlib, wall-clock timing through time, and complete error capture through traceback. Matplotlib creates the final single coverage chart, while pandas organises the coverage and summary tables as well as the per-clip findings. Because this notebook orchestrates an existing component rather than training anything of its own, the list is purposefully short.

In [ ]:
# import required libraries
import sys
import time
import traceback
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

The evaluation runs consistently every time it is carried out because the imports are explicit and collected in one location, which enables the results to be quoted as proof in the report. Machine-learning libraries are pulled in by the pipeline when it is loaded in the following section, thus there is no need to import them here directly because the heavy computation takes place inside the pipeline rather than in this notebook.

## 3 Load Solace Pipeline

This portion loads the multimodal pipeline itself and allows the application's own code to be imported from within the notebook. The MultimodalPipeline class, which is the exact object used by the live application to score a recording, is then imported and instantiated once the project root, which is situated two levels above the notebook, is added to the Python path. The evaluation's goal is to show that the deployed scoring works across languages, not that an approximation of it does, therefore loading the actual pipeline rather than a re-implementation is crucial.

In [ ]:
# project root
PROJECT_ROOT = Path.cwd().parents[1]

In [ ]:
# if project root not in path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# import the MultimodalPipeline class
from UI.ai.multimodal_pipeline import MultimodalPipeline

In [ ]:
# pipeline
pipeline = MultimodalPipeline()

In [ ]:
# print statements
print("Project root:", PROJECT_ROOT)
print("Solace pipeline loaded")

Every conclusion made in this notebook pertains to the system that users really engage with because the production pipeline is employed directly, which is what gives the functional evaluation its weight. The pipeline object and its underlying models are available, according to the printed confirmation. After obtaining the scorer, the following part modifies the study settings, which inform the notebook about the location of the recordings as well as the expected file kinds and languages.

## 4 Study Settings

The root folder containing the functional-evaluation recordings, the four available languages in a predetermined reporting sequence, and the video and audio file extensions that the pipeline can handle are all gathered in one location in this part. The recordings are arranged into one folder per language under a single root, and each folder may contain audio files, video files, or both. The pipeline treats the two differently, adding facial-emotion and visual behavioral signals only for video, so the notebook separates the two by file extension. The recording search is resilient regardless of the format in which a clip is saved when the acceptable extensions are explicitly defined.

In [ ]:
# recording directory
RECORDINGS_DIR = PROJECT_ROOT / "Functional Testing"

In [ ]:
# define the four supported languages
LANGUAGES = ["English", "Tamil", "Chinese", "Malay"]

In [ ]:
# define video file extensions
VIDEO_EXTENSIONS = [".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"]

In [ ]:
# define audio file extensions
AUDIO_EXTENSIONS = [".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg"]

In [ ]:
# print statements
print("Recordings folder:", RECORDINGS_DIR)

Because adding a language or moving the recordings only requires one modification, organising the folder, language list, and accepted extensions leaves the rest of the notebook free of hard-coded values and facilitates the study's expansion. The notebook can find the recordings that are truly accessible and report the coverage they offer once the settings have been adjusted.

## 5 Locate the Recordings

The recordings on disk are located in this section, and the coverage they offer for each language is reported. Instead of assuming a predetermined set of file names, it looks through each language folder and gathers all files whose extension indicates that they are audio or video recordings, noting for each clip which language it belongs to and whether it will be analysed as audio or video. After that, a coverage table is created that includes a list of all the clips found as well as a note indicating that no sample is available for any language that has no recordings at all. The fastest way to verify the multilingual distribution of the content being reviewed and to identify a missing or incorrectly filed recording while there is still time to fix it is to read this coverage before to any scoring.

In [ ]:
# define recording type for
def recording_type_for(path):
    # extension
    extension = path.suffix.lower()
    # a known video extension
    if extension in VIDEO_EXTENSIONS:
        return "video"
    # a known audio extension
    if extension in AUDIO_EXTENSIONS:
        return "audio"
    return None

In [ ]:
# define discover clips
def discover_clips():
    # empty dictionary
    found = {}
    for language in LANGUAGES:
        # path to this language folder
        folder = RECORDINGS_DIR / language
        if folder.is_dir():
            clips = sorted(p for p in folder.iterdir() if recording_type_for(p) is not None)
        else:
            clips = []
        # store clips discovered for this language
        found[language] = clips
    return found

In [ ]:
for language in LANGUAGES:
    # create folder
    (RECORDINGS_DIR / language).mkdir(parents=True, exist_ok=True)

In [ ]:
# discover recordings currently available
CLIPS = discover_clips()

In [ ]:
# coverage table
coverage_rows = []

In [ ]:
for language in LANGUAGES:
    # add one row per recording present for this language
    for clip in CLIPS[language]:
        coverage_rows.append({"language": language, "clip": clip.name, "type": recording_type_for(clip)})
    if not CLIPS[language]:
        coverage_rows.append({"language": language, "clip": "(no sample available)", "type": "-"})

In [ ]:
# coverage dataframe
coverage_df = pd.DataFrame(coverage_rows)
coverage_df

**Analysis:** 

An honest map of the content on which this judgment is based can be seen in the coverage table above. It identifies the precise recordings that were discovered for each language along with whether they will be analysed as audio or video. If there is no film for any language, it clearly states that no sample is available rather than letting that language disappear from the study. This allows the breadth of the functional evidence to be shown before a single model is run languages supported by actual recordings can legitimately support an operability claim, whereas a language without a sample is publicly recognised as untested until a recording is received. Since a recording saved under the incorrect language folder would be readily visible now, it is also the ideal time to find a misfiled clip. The notebook may now run the entire pipeline on each clip and note whether each stage provided the desired result after locating and classifying every accessible recording.

## 6 Score the Recordings with Solace

Every detected recording is subjected to the whole Solace pipeline in this part, which also notes whether or not each functional condition was met. The pipeline's own result dictionary and the recording type are returned by the helper, which executes the pipeline in the recording's native language and passes a progress callback to capture the stages reported by the pipeline as proof of processing advancement. The notebook uses that dictionary to determine whether a transcript was generated, whether text and audio scores were calculated, whether a facial score was generated for video clips, whether a fused wellbeing score was returned, whether a recommendation was generated, and whether the language path was correctly exercised. Every recording is handled inside a guard so that, instead of stopping the evaluation altogether, a single unreadable file communicates its fault and is skipped.

In [ ]:
# define run functional check
def run_functional_check(clip_path, language):
    # recording should be treated as video or audio
    recording_type = recording_type_for(clip_path)
    # stages list
    stages = []
    def progress(key):
        # stages append key
        stages.append(key)
    # pipeline analyse
    result = pipeline.analyse(str(clip_path), recording_type, progress, language_name=language)
    return result, recording_type

In [ ]:
# define present value
def _present(value):
    return value is not None and (not isinstance(value, str) or value.strip() != "")

In [ ]:
# functional outcomes per recording
functional_rows = []

In [ ]:
for language in LANGUAGES:
    for clip_path in CLIPS[language]:
        print(f"Scoring {clip_path.name} ({language})...")
        # start timing this recording
        start = time.time()
        try:
            # run pipeline
            result, recording_type = run_functional_check(clip_path, language)
            # transcript, text, audio, video scores produced
            transcript_ok = _present(result.get("transcript"))
            text_ok = _present(result.get("text_score"))
            audio_ok = _present(result.get("audio_score"))
            vision_ok = _present(result.get("vision_score")) if recording_type == "video" else True
            # wellbeing present
            wellbeing_ok = _present(result.get("wellbeing_score"))
            # recommendation produces
            recommendation_ok = _present(result.get("recommendation"))
            if language == "English":
                language_ok = (result.get("language") == "English") and recommendation_ok
            else:
                # language
                language_ok = (
                    result.get("language") == language
                    and _present(result.get("recommendation"))
                    and result.get("recommendation") != result.get("recommendation_english")
                )
            # funcitonal rows append
            functional_rows.append({
                "language": language,
                "clip": clip_path.name,
                "type": recording_type,
                "ran": True,
                "transcript": transcript_ok,
                "text": text_ok,
                "audio": audio_ok,
                "vision": vision_ok,
                "wellbeing": wellbeing_ok,
                "recommendation": recommendation_ok,
                "language_ok": language_ok,
                "wellbeing_score": result.get("wellbeing_score"),
                "seconds": round(time.time() - start, 2),
                "error": "",
            })
        # error
        except Exception as error:
            print(f"  Failed to score {clip_path.name}: {error}")
            # functional rows append to list
            functional_rows.append({
                "language": language, "clip": clip_path.name, "type": recording_type_for(clip_path),
                "ran": False, "transcript": False, "text": False, "audio": False, "vision": False,
                "wellbeing": False, "recommendation": False, "language_ok": False,
                "wellbeing_score": None, "seconds": round(time.time() - start, 2),
                "error": traceback.format_exc().splitlines()[-1],
            })

In [ ]:
for language in LANGUAGES:
    if not CLIPS[language]:
        # language append
        functional_rows.append({"language": language, "clip": "(no sample available)"})

In [ ]:
# functional dataframe
functional_df = pd.DataFrame(functional_rows)
functional_df.round(2)

**Analysis:** 

The functional evaluation's main piece of evidence is the table above, which provides an accurate description of the pipeline's output for each recording rather than a single collapsed verdict. A recording that partially succeeds for instance, an audio clip that is successfully transcribed and scored but lacks a facial score because it is audio is accurately represented rather than being tagged a failure for a stage that never applied to it since each criterion is reported independently. The wellbeing score recorded for each clip is the tangible result of that end-to-end run. For the languages supported by actual footage, it is expected that all relevant criteria are met, proving that transcription, the individual model scores, the fusion into a wellbeing score, and the translated recommendation all function in that language. Any language displaying no sample is carried through unchanged as an acknowledged gap, and any recording that failed is identified above with the last line of its mistake so that the cause can be investigated publicly rather than concealed. The following part condenses the per-recording detail into a per-language perspective of how fully the pipeline functioned because it is most useful when summarised by language.

## 7 Functional Coverage by Language

This section reports the percentage of each language's recordings that met each functional condition, summarising the per-recording data into a per-language view. Since each criterion is a boolean, average it across recordings for a language and reporting the outcome as a percentage provides a clear indication of how well the pipeline performed for that language. The same figures are then used to create a brief narrative that is filled with the actual run results rather than illustrative values, with the important statistics expressed in words next to the table.

In [ ]:
# list functional criteria to summarise
criteria = ["ran", "transcript", "text", "audio", "vision", "wellbeing", "recommendation", "language_ok"]

In [ ]:
# keep only recordings that were evaluated
evaluated = functional_df[functional_df["clip"] != "(no sample available)"]

In [ ]:
# summarise pass rate per language when there is anything to summarise
if len(evaluated) > 0:
    summary = evaluated.groupby("language")[criteria].mean().mul(100).round(0).astype("Int64")
    summary = summary.reindex(LANGUAGES)
    summary

In [ ]:
# define build analysis
def build_analysis():
    if len(evaluated) == 0:
        return "No recordings were evaluated, so no functional coverage can be reported yet."
    # total count
    total = len(evaluated)
    ran = int(evaluated["ran"].sum())
    lines = [f"A total of {total} recording(s) were evaluated, of which {ran} ran end to end without error."]
    for language in LANGUAGES:
        # select language evaluated recordings
        sub = evaluated[evaluated["language"] == language]
        if len(sub) == 0:
            lines.append(f"{language}: no sample available, so no functional evidence is reported.")
            continue
        # count the key
        n = len(sub)
        transcript = int(sub["transcript"].sum())
        wellbeing = int(sub["wellbeing"].sum())
        recommendation = int(sub["recommendation"].sum())
        language_ok = int(sub["language_ok"].sum())
        # compute mean wellbeing score where it exists
        mean_wb = sub["wellbeing_score"].dropna().mean()
        wb_text = f"{mean_wb:.1f}" if pd.notna(mean_wb) else "n/a"
        # lines append to list
        lines.append(
            f"{language} (n={n}): transcript produced for {transcript} of {n}, "
            f"wellbeing score for {wellbeing} of {n}, recommendation for {recommendation} of {n}, "
            f"language path correct for {language_ok} of {n}; mean wellbeing score {wb_text}."
        )
    return "\n".join(lines)

In [ ]:
# print analysis
print(build_analysis())

**Analysis:** 

Using numbers straight from the run, the summary and narrative it produces transform the per-recording detail into a claim that can be presented in the report. A high or full pass rate in transcription, scoring, fusion, recommendation, and language check for the languages covered by actual video provides convincing proof that the multilingual pipeline is actually functional for those languages, not just installed. Malay, in which there is no recording, contains no functional evidence and is presented as such Chinese, supported by a single recording, is better understood as an operability check that verifies the Chinese path executes rather than as a robustness assertion. The proper criteria for a functional evaluation of this size is to read the pass rates next to the number of recordings that support them in order to maintain the claim's strength proportionate to the evidence. The final section directly visualises the per-language pass rate because a single figure conveys the coverage more quickly than a table.

## 8 Coverage Visualisation

This section provides a single view of which languages are completely exercised, which are just partially covered, and which have no data by plotting the mean functional pass rate for each language across all criteria. In keeping with the truthful reporting of coverage that permeates the entire notebook, a language with no recording is purposefully represented as a labeled empty bar rather than removed so that the figure itself communicates both what has been demonstrated and what is still pending.

In [ ]:
# draw chart only when there are evaluated recordings
if len(evaluated) > 0:
    # mean pass rate per language across all criteria
    overall = evaluated.groupby("language")[criteria].mean().mean(axis=1).mul(100).reindex(LANGUAGES)
    heights = overall.fillna(0).values
    fig, ax = plt.subplots(figsize=(7, 5))
    # bar chart
    bars = ax.bar(LANGUAGES, heights)
    ax.set_ylabel("Mean functional pass rate (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Solace pipeline functional coverage by language")
    for bar, language in zip(bars, LANGUAGES):
        # write no data
        if pd.isna(overall[language]):
            ax.text(bar.get_x() + bar.get_width() / 2, 3, "no data", ha="center", va="bottom", rotation=90)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{bar.get_height():.0f}%", ha="center", va="bottom")
    plt.show()

**Analysis:** 

The chart makes the coverage story legible at a glance. Tall bars for the well-covered languages indicate that the pipeline functions completely in those languages an empty bar, on the other hand, indicates a language for which functional evidence has yet to be obtained rather than a silent omission. When presented in this manner, the visualisation supports a confident but limited claim the pipeline is clearly functional for the languages having recordings, and the effort required to finish the image acquiring a Malay recording and more Chinese recordings is equally obvious. In a study of size, the figure frequently provides a more comprehensive overview of the functional position than any one figure.

## 9 Conclusion

The Solace multimodal pipeline has been functionally evaluated in this notebook using all four of the supported languages. Before summarising the results by language and visualising the coverage, it found the actual recordings that were available for each language, ran each one through the precise production pipeline that the application uses, and recorded for each recording whether transcription, the text, audio, and facial models, the fusion into a wellbeing score, and the translated recommendation each produced their expected output.

The per-language pass rate and the quantity of recordings that support it are the evidence to be read from a finished run. Since the entire chain from voice to a language-appropriate recommendation is tested, a high pass rate for the languages supported by real video is compelling proof that the multilingual pipeline is functional rather than just existing in the code. Any language listed as having no sample is an accepted gap that the notebook automatically fixes once a recording is uploaded and the evaluation is rerun. Chinese, which is based on a single recording, is an operability check rather than a robustness claim.

The main drawback is coverage rather than methodology the functional technique is good and applies to all languages equally, but the number of recordings supporting each language determines how strong the assertion is. Its honesty includes framing the outcome in this manner. The notebook shows how the pipeline works multilingually on the available material, clearly indicates where that material is lacking or thin, and offers a repeatable process whose results become stronger as more recordings are acquired.

## 10 References

[1] A. Radford, J. W. Kim, T. Xu, G. Brockman, C. McLeavey, and I. Sutskever, "Robust speech recognition via large-scale weak supervision," OpenAI, 2022. [Online]. Available: https://arxiv.org/abs/2212.04356. [Accessed: 27 June 2026].

[2] NLLB Team et al., "No language left behind: scaling human-centered machine translation," Meta AI, 2022. [Online]. Available: https://arxiv.org/abs/2207.04672. [Accessed: 27 June 2026].